In [3]:
import pandas as pd
from datetime import datetime

data = pd.read_csv(r"../datasets/train.csv")

education_order = {"unknown": 0, "primary": 1, "secondary": 2, "tertiary": 3}
data["education"] = data["education"].map(education_order)
data['job'] = data['job'].str.replace('.', '', regex=False)
data = pd.get_dummies(data, columns=['marital', "contact", "poutcome", "job"], dtype="int")
data["default"] = data["default"].map({"no": 0, "yes": 1})
data["housing"] = data["housing"].map({"no": 0, "yes": 1})
data["loan"] = data["loan"].map({"no": 0, "yes": 1})
data["month"] = data["month"].apply(lambda x: datetime.strptime(x.strip().title(), "%b").month)

In [53]:
data.head()

,id,age,education,default,balance,housing,loan,day,month,duration,...,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,job_unknown
0,0,42,2,0,7,0,0,25,8,117,...,0,0,0,0,0,0,0,1,0,0
1,1,38,2,0,514,0,0,18,6,185,...,0,0,0,0,0,0,0,0,0,0
2,2,36,2,0,602,1,0,14,5,111,...,0,0,0,0,0,0,0,0,0,0
3,3,27,2,0,34,1,0,28,5,10,...,0,0,0,0,0,0,1,0,0,0
4,4,26,2,0,889,1,0,3,2,902,...,0,0,0,0,0,0,0,1,0,0


In [54]:
data.columns

Index(['id', 'age', 'education', 'default', 'balance', 'housing', 'loan',
       'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'y',
       'marital_divorced', 'marital_married', 'marital_single',
       'contact_cellular', 'contact_telephone', 'contact_unknown',
       'poutcome_failure', 'poutcome_other', 'poutcome_success',
       'poutcome_unknown', 'job_admin', 'job_blue-collar', 'job_entrepreneur',
       'job_housemaid', 'job_management', 'job_retired', 'job_self-employed',
       'job_services', 'job_student', 'job_technician', 'job_unemployed',
       'job_unknown'],
      dtype='object')

In [2]:
data.isna().sum()

NameError: name 'data' is not defined

In [56]:
data.dtypes

id                   int64
age                  int64
education            int64
default              int64
balance              int64
housing              int64
loan                 int64
day                  int64
month                int64
duration             int64
campaign             int64
pdays                int64
previous             int64
y                    int64
marital_divorced     int64
marital_married      int64
marital_single       int64
contact_cellular     int64
contact_telephone    int64
contact_unknown      int64
poutcome_failure     int64
poutcome_other       int64
poutcome_success     int64
poutcome_unknown     int64
job_admin            int64
job_blue-collar      int64
job_entrepreneur     int64
job_housemaid        int64
job_management       int64
job_retired          int64
job_self-employed    int64
job_services         int64
job_student          int64
job_technician       int64
job_unemployed       int64
job_unknown          int64
dtype: object

In [57]:
data.shape

(750000, 36)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

x = data.drop(columns=["y"])
y = data["y"]

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
accuracy_list, precision_list, recall_list, f1_list = [], [], [], []

for fold, (train_idx, test_idx) in enumerate(skf.split(x, y), 1):
    X_train, X_test = x.iloc[train_idx], x.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    accuracy_list.append(acc)
    precision_list.append(prec)
    recall_list.append(rec)
    f1_list.append(f1)

    print(f"🔹 Fold {fold} Results:")
    print(f"   Accuracy:  {acc:.3f}")
    print(f"   Precision: {prec:.3f}")
    print(f"   Recall:    {rec:.3f}")
    print(f"   F1 Score:  {f1:.3f}\n")

# ================== 5️⃣ Average Metrics ==================
print("🔸 Average Cross-Validation Results:")
print(f"✅ Mean Accuracy:  {np.mean(accuracy_list):.3f}")
print(f"🎯 Mean Precision: {np.mean(precision_list):.3f}")
print(f"📈 Mean Recall:    {np.mean(recall_list):.3f}")
print(f"💥 Mean F1 Score:  {np.mean(f1_list):.3f}")
